In [26]:
import os
import json
import fitz #(PyMuPDF)
from tqdm import tqdm
import shutil
import json
import pandas as pd
import csv
import re

La méthode présentée permet de sauvegarder plus facilement les résultats au format json, en séparant les informations de chaque page du document pdf. Étant donné que l'invite prévue pour le traitement ultérieur du résultat implique de travailler sur un fichier csv, nous allons d'abord extraire le document json, puis le fichier csv.

!! Dans les situations quand pendant le traitement de fichier fitz(PyMuPDF) retourne un erreur, ce fichier est transefé dans un autre dossier. Ce sont probablement les fichier dont le texte est impossible pour l'extraction par ce methode - il faudra les traiter par le methode 2. 

In [11]:
base_dir = os.path.expanduser("~/Documents/hackaton_week/ScincePo_data_copy")
out_jsonl = os.path.join(base_dir, "pdf_pages_extracted.jsonl")

errors_dir = os.path.join(base_dir, "_errors")
os.makedirs(errors_dir, exist_ok=True)

#On indique les dossiers qu'on veut ignorer : dans ce cas ce sont les dossiers avec les fichiers _falc et les fichiers PDF-images sans text déjà extraits
skip_names = {"_falc"}

In [15]:

# Désactiver l'affichage des avertissements et erreurs MuPDF
fitz.TOOLS.mupdf_display_errors(False)
fitz.TOOLS.mupdf_display_warnings(False)

# Valeur utilisée pour remplacer les cellules vides
EMPTY_CELL_VALUE = "n"

# Exclure le dossier _errors du parcours
skip_names.add("_errors")

# Fichier texte pour enregistrer les PDF provoquant des erreurs
error_txt = os.path.join(base_dir, "pdf_errors.txt")

#Constitution de la liste des fichiers PDF
pdf_paths = []
for root, dirs, files in os.walk(base_dir):
    # Exclure les dossiers non pertinents du parcours
    dirs[:] = [d for d in dirs if d not in skip_names]
    for f in files:
        if f.lower().endswith(".pdf"):
            pdf_paths.append(os.path.join(root, f))

pdf_paths = sorted(pdf_paths)

def normalize_cell(value: str | None) -> str:
    """
    Normalise les valeurs textuelles.
    Garantit qu'aucune chaîne vide n'est retournée.
    """
    v = (value or "").strip()
    return v if v else EMPTY_CELL_VALUE

#Identification des fichiers déjà traités
processed_filenames = set()
if os.path.exists(out_jsonl):
    with open(out_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                fn = obj.get("filename")
                if fn:
                    processed_filenames.add(fn)
            except json.JSONDecodeError:
                # Ignorer les lignes corrompues du fichier JSONL
                continue

print(f"Nombre de fichiers PDF déjà traités : {len(processed_filenames)}")

#Chargement des erreurs déjà enregistrées
logged_errors = set()
if os.path.exists(error_txt):
    with open(error_txt, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                logged_errors.add(line)

def log_error_filename(filename: str):
    """
    Enregistre le nom d'un fichier PDF ayant provoqué une erreur
    dans un fichier texte dédié, sans duplication.
    """
    if filename in logged_errors:
        return
    with open(error_txt, "a", encoding="utf-8") as f:
        f.write(filename + "\n")
    logged_errors.add(filename)

# Liste des fichiers déplacés vers _errors lors de cette session
moved_files = []

#Extraction du texte à partir des PDF (PyMuPDF)
def extract_pages(pdf_path: str, filename: str) -> list[str]:
    """
    Extrait le texte d'un fichier PDF page par page à partir des blocs de texte.
    En cas d'erreur d'ouverture ou de lecture, le fichier est consigné
    dans un fichier texte d'erreurs et déplacé dans le dossier _errors.
    """
    pages_out = []
    try:
        doc = fitz.open(pdf_path)
        for page in doc:
            blocks = page.get_text("blocks")
            clean = []

            for x0, y0, x1, y1, text, *_ in blocks:
                t = (text or "").strip()
                if not t:
                    continue
                clean.append((y0, x0, t))

            # Réorganisation du texte selon la position spatiale
            clean.sort(key=lambda k: (k[0], k[1]))
            page_text = "\n\n".join(t for _, _, t in clean)
            pages_out.append(normalize_cell(page_text))

        return pages_out

    except Exception:
        # Enregistrer le fichier PDF ayant causé une erreur
        log_error_filename(filename)
        # Déplacer le fichier vers le dossier _errors
        dest = os.path.join(errors_dir, filename)
        try:
            shutil.move(pdf_path, dest)
            moved_files.append(filename)
        except Exception:
            pass
        return []

#Sauvegarde incrémentale au format JSONL
def flush_batch(rows: list[dict], jsonl_path: str):
    """
    Écrit un lot de résultats dans le fichier JSONL
    afin de limiter la perte de données en cas d'interruption.
    """
    if not rows:
        return
    with open(jsonl_path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

batch_rows = []
total_done = 0

#Boucle principale d'extraction (reprise possible)
for path in tqdm(pdf_paths, desc="Extraction du contenu des fichiers PDF", unit="pdf"):
    filename = os.path.basename(path)

    # Ignorer les fichiers déjà traités
    if filename in processed_filenames:
        continue

    pages = extract_pages(path, filename)

    # Garantir au moins une valeur non vide
    if not pages:
        pages = [EMPTY_CELL_VALUE]

    record = {
        "filename": normalize_cell(filename),
        "pages": [normalize_cell(p) for p in pages],
        "num_pages": len(pages),
    }

    batch_rows.append(record)
    processed_filenames.add(filename)
    total_done += 1

    # Sauvegarde périodique
    if total_done % 100 == 0:
        flush_batch(batch_rows, out_jsonl)
        batch_rows = []

# Écriture des résultats restants
flush_batch(batch_rows, out_jsonl)

print("Traitement terminé.")
print(f"Nombre de nouveaux fichiers PDF traités : {total_done}")
print(f"Fichier JSONL mis à jour : {out_jsonl}")
print(f"Liste des fichiers PDF en erreur : {error_txt}")
print(f"Nombre de fichiers déplacés vers _errors : {len(moved_files)}")


Nombre de fichiers PDF déjà traités : 11544


Extraction du contenu des fichiers PDF: 100%|██████████| 14631/14631 [00:49<00:00, 293.40pdf/s] 

Traitement terminé.
Nombre de nouveaux fichiers PDF traités : 3087
Fichier JSONL mis à jour : /Users/quentinnippert/Documents/hackaton_week/ScincePo_data_copy/pdf_pages_extracted.jsonl
Liste des fichiers PDF en erreur : /Users/quentinnippert/Documents/hackaton_week/ScincePo_data_copy/pdf_errors.txt
Nombre de fichiers déplacés vers _errors : 0


Exemple de résultat JSON - on voit ici que le code met l'information sur deux pages dans une seule cellule, en les divisant par ".," - on va utiliser cette information pour séparer les pages et transformer le fichier dans le format csv. 

In [ ]:
obj = pd.read_json("/Users/quentinnippert/Documents/hackaton_week/ScincePo_data_copy/pdf_pages_extracted.jsonl", lines = True)
obj

,filename,pages,num_pages
0,LG17-1-1-BONNOT-10-tour1-profession_foi.pdf,[Vincent GUERIN\n\nGilbert BONNOT\n\nSuppléant...,2
1,LG17-1-1-BRETON-5-tour1-profession_foi.pdf,"[xavier \nbreton»\n\nEngagé, \nà vos côtés\n\n...",2
2,LG17-1-1-BRETON-5-tour2-profession_foi.pdf,"[xavier \nbreton»\n\nEngagé, \nà vos côtés\n\n...",2
3,LG17-1-1-CARLIER-6-tour1-profession_foi.pdf,[ÉLECTIONS LÉGISLATIVES – 11 ET 18 JUIN 2017\n...,2
4,LG17-1-1-LÉPAGNOT-3-tour1-profession_foi.pdf,[ÉLECTIONS LÉGISLATIVES DE JUIN 2017\n1re circ...,2
...,...,...,...
14626,LG24-ZZ-ZZ08-tour1-9-Gilles NEFFATI_profession...,"[n, n]",2
14627,LG24-ZZ-ZZ08-tour2-6-Caroline YADAN_profession...,"[n, n]",2
14628,LG24-ZZ-ZZ09-tour1-6-Rachid TAHIRI_profession_...,"[n, n]",2
14629,LG24-ZZ-ZZ11-tour2-13-Anne GENETET_profession_...,"[n, n]",2


In [23]:
in_jsonl = "/Users/quentinnippert/Documents/hackaton_week/ScincePo_data_copy/pdf_pages_extracted.jsonl"

EMPTY = "n"
SEP = ".,"

df = pd.read_json(in_jsonl, lines=True)

def normalize_text(x: str) -> str:
    x = (x or "").strip()
    if not x:
        return EMPTY
    # чтобы CSV визуально не "ломался" от реальных переносов строк
    x = x.replace("\r\n", "\n").replace("\r", "\n").replace("\n", "\\n")
    return x

def split_pages_field(pages):
    """
    Возвращает list[str] страниц.
    - если pages уже list -> нормализуем элементы
    - если pages строка -> режем по SEP=".,"
    """
    if isinstance(pages, list):
        parts = [normalize_text(p) for p in pages]
        return parts if parts else [EMPTY]

    if pages is None:
        return [EMPTY]

    s = str(pages)
    # режем по точному маркеру между страницами
    raw_parts = [p.strip() for p in s.split(SEP)]
    raw_parts = [p for p in raw_parts if p]  # убрать пустые

    if not raw_parts:
        return [EMPTY]

    return [normalize_text(p) for p in raw_parts]

# 1) split
df["pages_list"] = df["pages"].apply(split_pages_field)

# 2) wide columns page_1..page_N
pages_wide = pd.DataFrame(df["pages_list"].tolist()).fillna(EMPTY)
pages_wide.columns = [f"page_{i+1}" for i in range(pages_wide.shape[1])]

# 3) DataFrame final (sans sauvegarde — sera fait après filtrage)
out_df = pd.concat([df[["filename"]].copy(), pages_wide], axis=1)
out_df["filename"] = out_df["filename"].astype(str).apply(normalize_text)
out_df = out_df.fillna(EMPTY)

print("Transformation terminée.")
print("Max pages:", pages_wide.shape[1])
print("Colonnes:", list(out_df.columns[:5]), "...")
print("Lignes:", len(out_df))


Transformation terminée.
Max pages: 7
Colonnes: ['filename', 'page_1', 'page_2', 'page_3', 'page_4'] ...
Lignes: 14631


In [24]:
out_df.head()

,filename,page_1,page_2,page_3,page_4,page_5,page_6,page_7
0,LG17-1-1-BONNOT-10-tour1-profession_foi.pdf,Vincent GUERIN\n\nGilbert BONNOT\n\nSuppléant\...,Ce que nous voulons\n\nDans une société dégrad...,n,n,n,n,n
1,LG17-1-1-BRETON-5-tour1-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe m’engage à encourager \nle TRAVAIL.\n...,n,n,n,n,n
2,LG17-1-1-BRETON-5-tour2-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe soutiendrai\nJe lutterai contre\n\nle ...,n,n,n,n,n
3,LG17-1-1-CARLIER-6-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES – 11 ET 18 JUIN 2017\nA...,LES CANDIDATS ET LE PROGRAMME DE L’UPR\n\nNos ...,n,n,n,n,n
4,LG17-1-1-LÉPAGNOT-3-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES DE JUIN 2017\n1re circo...,"Travailleuses, travailleurs,\n\nVous qui avez ...",n,n,n,n,n


Ici on supprime les pages >page_2 pour enlever l'information innécessaire. 

In [25]:
# Garder uniquement filename, page_1, page_2
cols_to_keep = ["filename", "page_1"]
if "page_2" in out_df.columns:
    cols_to_keep.append("page_2")
else:
    out_df["page_2"] = "n"
    cols_to_keep.append("page_2")

out_df = out_df[cols_to_keep].copy()

print("Colonnes conservées :", out_df.columns.tolist())
print("Lignes :", len(out_df))

out_df.head()

Colonnes conservées : ['filename', 'page_1', 'page_2']
Lignes : 14631


,filename,page_1,page_2
0,LG17-1-1-BONNOT-10-tour1-profession_foi.pdf,Vincent GUERIN\n\nGilbert BONNOT\n\nSuppléant\...,Ce que nous voulons\n\nDans une société dégrad...
1,LG17-1-1-BRETON-5-tour1-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe m’engage à encourager \nle TRAVAIL.\n...
2,LG17-1-1-BRETON-5-tour2-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe soutiendrai\nJe lutterai contre\n\nle ...
3,LG17-1-1-CARLIER-6-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES – 11 ET 18 JUIN 2017\nA...,LES CANDIDATS ET LE PROGRAMME DE L’UPR\n\nNos ...
4,LG17-1-1-LÉPAGNOT-3-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES DE JUIN 2017\n1re circo...,"Travailleuses, travailleurs,\n\nVous qui avez ..."


Nettoyage du texte inutile mis automatiquement par modèle PyMuPDF

In [27]:
# Nettoyage
text_cols = out_df.columns[1:]

# Supprime la mention TCPDF (tolérant aux variations d'espaces et casse)
pattern = re.compile(r"Powered\s+by\s+TCPDF\s*\(www\.tcpdf\.org\)", flags=re.IGNORECASE)

for col in text_cols:
    out_df[col] = (
        out_df[col]
        .astype("string")
        .str.replace(pattern, "", regex=True)
        .str.strip()
        .fillna("n")
    )

print("Nettoyage terminé (out_df).")
print("Colonnes:", out_df.columns.tolist())
print("Lignes:", len(out_df))
out_df.head()

Nettoyage terminé (out_df).
Colonnes: ['filename', 'page_1', 'page_2']
Lignes: 14631


,filename,page_1,page_2
0,LG17-1-1-BONNOT-10-tour1-profession_foi.pdf,Vincent GUERIN\n\nGilbert BONNOT\n\nSuppléant\...,Ce que nous voulons\n\nDans une société dégrad...
1,LG17-1-1-BRETON-5-tour1-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe m’engage à encourager \nle TRAVAIL.\n...
2,LG17-1-1-BRETON-5-tour2-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe soutiendrai\nJe lutterai contre\n\nle ...
3,LG17-1-1-CARLIER-6-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES – 11 ET 18 JUIN 2017\nA...,LES CANDIDATS ET LE PROGRAMME DE L’UPR\n\nNos ...
4,LG17-1-1-LÉPAGNOT-3-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES DE JUIN 2017\n1re circo...,"Travailleuses, travailleurs,\n\nVous qui avez ..."


Au cas où meme si après le traitement les fichiers qui renvoient l'erreur après le traitement par PyMuPDF il reste des fichiers mal traités par ce méthode, ce code permet de chercher ces fichers pour les traiter egalement par le methode 2. 

Critères de recherche : 
- Une des pages est vide
- Texte suspect de type “lettres aléatoires / capslock”
- Absence totale de caractères alphanumériques
- Seulement un seul mot détecté

In [31]:
"""
Contrôle qualité du texte + déplacement des PDF invalides vers _errors.
En sortie : sauvegarde du CSV final sans lignes invalides.
"""

# Travailler directement sur out_df (en mémoire)
df = out_df.copy()

# Sélection des colonnes textuelles
text_columns = df.columns[1:]

# Expressions régulières
word_re = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿА-Яа-я]{2,}")
alnum_re = re.compile(r"[0-9A-Za-zÀ-ÖØ-öø-ÿА-Яа-я]")
single_letter_re = re.compile(r"^[A-Za-zÀ-ÖØ-öø-ÿА-Яа-я]$")
empty_markers = {"n", "nan", "none", "null"}


def est_colonne_invalide(val) -> bool:
    """Détermine si une cellule textuelle est invalide."""
    if pd.isna(val):
        return True

    s = str(val).strip()
    s_lower = s.lower()

    # Marqueurs de vide explicites
    if s == "" or s_lower in empty_markers:
        return True

    # Règle demandée : 1 seule lettre => invalide
    if single_letter_re.fullmatch(s):
        return True

    if not alnum_re.search(s):
        return True

    mots = word_re.findall(s)
    if len(mots) == 1:
        return True

    letters_only = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿА-Яа-я]", "", s)
    if letters_only:
        upper_ratio = sum(ch.isupper() for ch in letters_only) / len(letters_only)
        if len(mots) < 2 and len(letters_only) <= 30 and upper_ratio >= 0.8:
            return True

    return False


def ligne_a_sortir(row) -> bool:
    for col in text_columns:
        if est_colonne_invalide(row[col]):
            return True
    return False


# 1) Détection des lignes invalides
masque_sortir = df.apply(ligne_a_sortir, axis=1)
df_a_sortir = df[masque_sortir].copy()
df_reste = df[~masque_sortir].copy()

print("Nombre total de lignes :", len(df))
print("Nombre de lignes isolées (contenu invalide) :", len(df_a_sortir))
print("Nombre de lignes conservées :", len(df_reste))


# 2) Déplacement des PDF invalides vers _errors
copy_base_dir = os.path.expanduser("~/Documents/hackaton_week/ScincePo_data_copy")
search_folders = ["LG17", "LG20", "LG22", "LG24"]
errors_dir = os.path.join(copy_base_dir, "_errors")
os.makedirs(errors_dir, exist_ok=True)

invalid_filenames = (
    df_a_sortir["filename"]
    .dropna()
    .astype(str)
    .str.strip()
)
invalid_filenames = [f for f in invalid_filenames if f and f.lower() != "n"]
invalid_filenames = sorted(set(invalid_filenames))

moved_count = 0
not_found_count = 0
already_in_errors = 0

for filename in invalid_filenames:
    src_path = None

    # Si le fichier est déjà dans _errors, ne rien faire
    if os.path.exists(os.path.join(errors_dir, filename)):
        already_in_errors += 1
        continue

    # Recherche du fichier par nom dans LG17 / LG20 / LG22 / LG24
    for folder in search_folders:
        root_folder = os.path.join(copy_base_dir, folder)
        if not os.path.isdir(root_folder):
            continue

        for root, dirs, files in os.walk(root_folder):
            if filename in files:
                src_path = os.path.join(root, filename)
                break

        if src_path:
            break

    if not src_path:
        not_found_count += 1
        print(f"[NON TROUVÉ] {filename}")
        continue

    dst_path = os.path.join(errors_dir, filename)

    # Évite d'écraser un fichier existant
    if os.path.exists(dst_path):
        base, ext = os.path.splitext(filename)
        i = 1
        while True:
            candidate = os.path.join(errors_dir, f"{base}__dup{i}{ext}")
            if not os.path.exists(candidate):
                dst_path = candidate
                break
            i += 1

    shutil.move(src_path, dst_path)
    moved_count += 1

print("\nDéplacement des fichiers invalides terminé.")
print("Fichiers invalides uniques analysés :", len(invalid_filenames))
print("Déplacés vers _errors :", moved_count)
print("Déjà présents dans _errors :", already_in_errors)
print("Introuvables :", not_found_count)


# 3) Sauvegarde du CSV final (sans lignes invalides)
out_csv_pdf = os.path.join(copy_base_dir, "csv_pdf_final.csv")
df_reste.to_csv(out_csv_pdf, index=False, encoding="utf-8")

print("\nCSV final sauvegardé :", out_csv_pdf)
print("Lignes dans le CSV final :", len(df_reste))

Nombre total de lignes : 14631
Nombre de lignes isolées (contenu invalide) : 3962
Nombre de lignes conservées : 10669

Déplacement des fichiers invalides terminé.
Fichiers invalides uniques analysés : 3962
Déplacés vers _errors : 2725
Déjà présents dans _errors : 1237
Introuvables : 0

CSV final sauvegardé : /Users/quentinnippert/Documents/hackaton_week/ScincePo_data_copy/csv_pdf_final.csv
Lignes dans le CSV final : 10669


In [34]:
df_reste.head()

,filename,page_1,page_2
0,LG17-1-1-BONNOT-10-tour1-profession_foi.pdf,Vincent GUERIN\n\nGilbert BONNOT\n\nSuppléant\...,Ce que nous voulons\n\nDans une société dégrad...
1,LG17-1-1-BRETON-5-tour1-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe m’engage à encourager \nle TRAVAIL.\n...
2,LG17-1-1-BRETON-5-tour2-profession_foi.pdf,"xavier \nbreton»\n\nEngagé, \nà vos côtés\n\nÉ...",»\n\nJe soutiendrai\nJe lutterai contre\n\nle ...
3,LG17-1-1-CARLIER-6-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES – 11 ET 18 JUIN 2017\nA...,LES CANDIDATS ET LE PROGRAMME DE L’UPR\n\nNos ...
4,LG17-1-1-LÉPAGNOT-3-tour1-profession_foi.pdf,ÉLECTIONS LÉGISLATIVES DE JUIN 2017\n1re circo...,"Travailleuses, travailleurs,\n\nVous qui avez ..."
